In [1]:
import pandas as pd
import json
import re

In [2]:
df = pd.read_csv('bills_to_labels.csv')

In [3]:
df.head()

,title,status,proposal_date,categories,creators
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,"[{""memberships"":[{""posts"":[{""label"":""เลขาธิการ..."
1,ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายวิธ...,IN_PROGRESS,2025-08-19,NaN,[{}]
2,ร่างพระราชบัญญัติกองทุนทุเรียนไทย พ.ศ. ....,REJECTED,2024-09-30,NaN,[]
3,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,"[{""memberships"":[{""posts"":[{""label"":""สมาชิกพรร..."
4,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,NaN,"[{""memberships"":[{""posts"":[{""label"":""หัวหน้าพร..."


In [4]:
df['title'].nunique()

441

In [5]:
# นับจำนวนแถวต่อ 1 title และเรียงจากมากไปน้อย
duplicate_counts = df.groupby('title').size().sort_values(ascending=False)

# แสดงเฉพาะอันที่ซ้ำ (count > 1)
print(duplicate_counts[duplicate_counts > 1])

title
ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายอาญา (ฉบับที่ ..) พ.ศ. ....                   17
ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดการประมง พ.ศ. 2558 พ.ศ. ....                   14
ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายวิธีพิจารณาความอาญา (ฉบับที่ ..) พ.ศ. ....    11
ร่างพระราชบัญญัติการศึกษาแห่งชาติ พ.ศ. ....                                              10
พระราชบัญญัติอ้อยและน้ำตาลทราย (ฉบับที่ ..) พ.ศ. ....                                     8
                                                                                         ..
ร่างพระราชบัญญัติรับราชการทหาร พ.ศ. ....                                                  2
ร่างพระราชบัญญัติระเบียบบริหารราชการฝ่ายรัฐสภา (ฉบับที่ ..) พ.ศ. ....                     2
ร่างพระราชบัญญัติธรรมนูญศาลทหาร (ฉบับที่่ ..) พ.ศ. ....                                   2
ร่างพระราชบัญญัติป่าไม้ (ฉบับที่ ..) พ.ศ. ....                                            2
ร่างพระราชบัญญัติกองทุนเงินให้กู้ยืมเพื่อการศึกษา (ฉบับที่ ..) พ.ศ. ....  

In [6]:
def extract_labels(json_str):
    try:
        data = json.loads(json_str)
        labels = []
        for entry in data:
            for membership in entry.get('memberships', []):
                for post in membership.get('posts', []):
                    label = post.get('label')
                    if label:
                        labels.append(label)
        return labels
    except:
        return []

df['labels_list'] = df['creators'].apply(extract_labels)

df_exploded = df.explode('labels_list').reset_index(drop=True)

df_result = df_exploded.drop(columns=['creators']).rename(columns={'labels_list': 'creator_label'})

print(df_result[['title', 'creator_label']].head(10))

                                               title  \
0  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
1  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
2  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
3  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
4  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
5  ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
6  ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายวิธ...   
7        ร่างพระราชบัญญัติกองทุนทุเรียนไทย พ.ศ. ....   
8    ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....   
9    ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....   

                                       creator_label  
0                          เลขาธิการพรรคประชาธิปัตย์  
1                          สมาชิกพรรครวมไทยสร้างชาติ  
2                             สมาชิกพรรคประชาธิปัตย์  
3                                      สส. ชุดที่ 25  
4  รัฐมนตรีว่าการกระทรวงการพัฒนาสังคมและความมั่นค...  
5                                      สส. ชุดที่ 26 

In [7]:
df_result.head(10)

,title,status,proposal_date,categories,creator_label
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,เลขาธิการพรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรคประชาธิปัตย์
3,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สส. ชุดที่ 25
4,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,รัฐมนตรีว่าการกระทรวงการพัฒนาสังคมและความมั่นค...
5,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สส. ชุดที่ 26
6,ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายวิธ...,IN_PROGRESS,2025-08-19,NaN,NaN
7,ร่างพระราชบัญญัติกองทุนทุเรียนไทย พ.ศ. ....,REJECTED,2024-09-30,NaN,NaN
8,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคก้าวไกล
9,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคอนาคตใหม่


In [8]:
df_result.shape[0]

2398

In [9]:
df_result = df_result.dropna(subset="creator_label")

In [10]:
df_result.head(10)

,title,status,proposal_date,categories,creator_label
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,เลขาธิการพรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรคประชาธิปัตย์
3,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สส. ชุดที่ 25
4,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,รัฐมนตรีว่าการกระทรวงการพัฒนาสังคมและความมั่นค...
5,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สส. ชุดที่ 26
8,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคก้าวไกล
9,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคอนาคตใหม่
10,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคประชาชน
11,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สส. ชุดที่ 26


In [11]:
df_result.shape[0]

2231

In [12]:
party_list = [
    'พรรคประชาธิปัตย์', 'พรรคประชากรไทย', 'พรรคความหวังใหม่', 
    'พรรคเครือข่ายชาวนาแห่งประเทศไทย', 'พรรคเพื่อไทย', 'พรรคชาติพัฒนา', 
    'พรรคชาติไทยพัฒนา', 'พรรคอนาคตไทย', 'พรรคภูมิใจไทย', 
    'พรรคสังคมประชาธิปไตยไทย', 'พรรครักชาติ', 'พรรคประชาธิปไตยใหม่', 
    'พรรคพลังบูรพา', 'พรรคครูไทยเพื่อประชาชน', 'พรรคพลังท้องถิ่นไท', 
    'พรรคประชาชน', 'พรรคไทยก้าวใหม่', 'พรรคเสรีรวมไทย', 
    'พรรครักษ์ธรรม', 'พรรคพลังประชาธิปไตย', 'พรรคพลังสุราษฎร์', 
    'พรรคพลังไทยรักชาติ', 'พรรคเพื่อชีวิตใหม่', 'พรรคทางเลือกใหม่', 
    'พรรคเศรษฐกิจ', 'พรรคสร้างอนาคตไทย', 'พรรคพลังธรรมใหม่', 
    'พรรคไทยธรรม', 'พรรครวมพลัง', 'พรรคไทยพร้อม', 
    'พรรคปวงชนไทย', 'พรรคเพื่อชาติไทย', 'พรรคก้าวอิสระ', 
    'พรรคประชาชาติ', 'พรรคแผ่นดินธรรม', 'พรรคคลองไทย', 
    'พรรคพลังประชารัฐ', 'พรรคเศรษฐกิจใหม่', 'พรรคพลังสังคม', 
    'พรรคเป็นธรรม', 'พรรคพลังเพื่อไทย', 'พรรคประชาไทย', 
    'พรรคกรีน', 'พรรควิชชั่นใหม่', 'พรรคพลวัต', 
    'พรรคกล้าธรรม', 'พรรคไทยรวมไทย', 'พรรคกล้า', 
    'พรรคฟิวชัน', 'พรรคพลังสังคมใหม่', 'พรรคไทยสร้างไทย', 
    'พรรคมิติใหม่', 'พรรครวมไทยสร้างชาติ', 'พรรคไทยสมาร์ท', 
    'พรรคไทยภักดี', 'พรรคไทยพิทักษ์ธรรม', 'พรรคไทยชนะ', 
    'พรรคไทรวมพลัง', 'พรรคราษฎร์วิถี', 'พรรคโอกาสใหม่', 
    'พรรคท้องที่ไทย', 'พรรคใหม่', 'พรรคแรงงานสร้างชาติ', 
    'พรรคไทยก้าวหน้า', 'พรรคตะวันใหม่', 'พรรคพร้อม', 
    'พรรครวมใจไทย', 'พรรคสัมมาธิปไตย', 'พรรครักภูเก็ต', 
    'พรรคประชาอาสาชาติ', 'พรรคไทยทรัพย์ทวี', 'พรรครวมพลังประชาชน', 
    'พรรคอนาคตไกล', 'พรยยางพาราไทย', 'พรรคเพื่อบ้านเมือง'
]

In [13]:
# 1. เตรียมรายชื่อพรรค (เอาเฉพาะชื่อหลักมาทำเป็น pattern)
# หมายเหตุ: ใช้ชื่อที่คุณส่งมา 75 พรรค โดยเน้นคำที่เป็น Keyword สำคัญ
party_pattern = '|'.join(party_list)

# 2. กรองข้อมูล: เก็บเฉพาะแถวที่ creator_label มีคำที่ตรงกับใน party_list
# เราใช้ na=False เพื่อข้ามค่าที่เป็น NaN
df_cleaned = df_result[df_result['creator_label'].str.contains(party_pattern, na=False)]

# 3. ตรวจสอบผลลัพธ์
print(f"จำนวนแถวก่อน Clean: {len(df_result)}")
print(f"จำนวนแถวหลัง Clean: {len(df_cleaned)}")
print("\nตัวอย่างข้อมูลที่ผ่านการ Clean:")
print(df_cleaned[['title', 'creator_label']].head(10))

จำนวนแถวก่อน Clean: 2231
จำนวนแถวหลัง Clean: 666

ตัวอย่างข้อมูลที่ผ่านการ Clean:
                                                title  \
0   ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
1   ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
2   ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...   
10    ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....   
13  ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...   
14  ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...   
15  ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...   
18  ร่างพระราชบัญญัติการส่งเสริมวิทยาศาสตร์ การวิจ...   
21     ร่างพระราชบัญญัติโรงงาน (ฉบับที่ ..) พ.ศ. ....   
25  ร่างพระราชบัญญัติธรรมนูญศาลทหาร (ฉบับที่่ ..) ...   

                    creator_label  
0       เลขาธิการพรรคประชาธิปัตย์  
1       สมาชิกพรรครวมไทยสร้างชาติ  
2          สมาชิกพรรคประชาธิปัตย์  
10              สมาชิกพรรคประชาชน  
13  หัวหน้าพรรคครูไทยเพื่อประชาชน  
14      กรรมการบริหารพรรคกล้าธรรม  
15   สมาชิกพรรคครูไทยเพื่อประชาชน  
18

In [14]:
# ดูว่าอันที่ไม่ผ่านเกณฑ์ (Dropped) มีคำว่าอะไรบ้าง
dropped_labels = df_result[~df_result['creator_label'].str.contains(party_pattern, na=False)]
print(dropped_labels['creator_label'].unique())

['สส. ชุดที่ 25'
 'รัฐมนตรีว่าการกระทรวงการพัฒนาสังคมและความมั่นคงของมนุษย์ ครม. คณะที่ 62'
 'สส. ชุดที่ 26' 'สมาชิกพรรคก้าวไกล' 'สมาชิกพรรคอนาคตใหม่'
 'รัฐมนตรีช่วยว่าการกระทรวงคมนาคม ครม. คณะที่ 65'
 'รัฐมนตรีว่าการกระทรวงอุตสาหกรรม ครม. คณะที่ 64'
 'รัฐมนตรีช่วยว่าการกระทรวงคมนาคม ครม. คณะที่ 63'
 'รัฐมนตรีช่วยว่าการกระทรวงคมนาคม ครม. คณะที่ 64'
 'รัฐมนตรีว่าการกระทรวงแรงงาน ครม. คณะที่ 65'
 'รัฐมนตรีว่าการกระทรวงศึกษาธิการ ครม. คณะที่ 62'
 'นายกรัฐมนตรี ครม. คณะที่ 65'
 'รัฐมนตรีว่าการกระทรวงมหาดไทย ครม. คณะที่ 65'
 'รองนายกรัฐมนตรี ครม. คณะที่ 62'
 'รัฐมนตรีว่าการกระทรวงมหาดไทย ครม. คณะที่ 64'
 'รองนายกรัฐมนตรี ครม. คณะที่ 63'
 'รัฐมนตรีว่าการกระทรวงสาธารณสุข ครม. คณะที่ 62'
 'รองนายกรัฐมนตรี ครม. คณะที่ 64'
 'รัฐมนตรีว่าการกระทรวงมหาดไทย ครม. คณะที่ 63'
 'รัฐมนตรีช่วยว่าการกระทรวงเกษตรและสหกรณ์ ครม. คณะที่ 65'
 'รองประธาน คนที่ 2 สส. ชุดที่ 26' 'รองประธาน คนที่ 1 สส. ชุดที่ 26'
 'รัฐมนตรีว่าการกระทรวงการท่องเที่ยวและกีฬา ครม. คณะที่ 64'
 'หัวหน้าพรรคก้าวไกล' 'เลขาธิการพรรคก้าวไกล

In [15]:
# สมมติว่า party_list ของคุณคือ list พรรค 75 พรรคที่คุยกันก่อนหน้า
# เพื่อความแม่นยำ ผมจะเรียงพรรคที่มีชื่อยาวกว่าไว้ก่อน (Prevent substring overlap)
sorted_parties = sorted(party_list, key=len, reverse=True)

def merge_to_party_name(label):
    if pd.isna(label):
        return label
    
    for party in sorted_parties:
        # ถ้าใน label (เช่น 'เลขาธิการพรรคประชาธิปัตย์') มีชื่อพรรคอยู่
        if party in str(label):
            return party  # คืนค่าเป็นชื่อพรรคเพียวๆ (เช่น 'พรรคประชาธิปัตย์')
            
    return label # ถ้าไม่เจอพรรคเลย ให้คืนค่าเดิมไว้ก่อน (เช่น สส. ชุดที่ 26)

# 1. รันฟังก์ชันเพื่อยุบรวมชื่อ
df_cleaned['party_cleaned'] = df_cleaned['creator_label'].apply(merge_to_party_name)

# 2. ตรวจสอบผลลัพธ์
print(df_cleaned[['creator_label', 'party_cleaned']].head(10))

                    creator_label           party_cleaned
0       เลขาธิการพรรคประชาธิปัตย์        พรรคประชาธิปัตย์
1       สมาชิกพรรครวมไทยสร้างชาติ     พรรครวมไทยสร้างชาติ
2          สมาชิกพรรคประชาธิปัตย์        พรรคประชาธิปัตย์
10              สมาชิกพรรคประชาชน             พรรคประชาชน
13  หัวหน้าพรรคครูไทยเพื่อประชาชน  พรรคครูไทยเพื่อประชาชน
14      กรรมการบริหารพรรคกล้าธรรม            พรรคกล้าธรรม
15   สมาชิกพรรคครูไทยเพื่อประชาชน  พรรคครูไทยเพื่อประชาชน
18             สมาชิกพรรคเพื่อไทย            พรรคเพื่อไทย
21              สมาชิกพรรคประชาชน             พรรคประชาชน
25            สมาชิกพรรคประชาชาติ           พรรคประชาชาติ


/var/folders/65/6rnggm5s6y5bnwcrgm55lbbw0000gn/T/ipykernel_41739/1229241951.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['party_cleaned'] = df_cleaned['creator_label'].apply(merge_to_party_name)


In [20]:
df_cleaned['party_cleaned'].value_counts()

พรรคประชาชน               178
พรรคภูมิใจไทย             114
พรรคเพื่อไทย              106
พรรคประชาธิปัตย์           77
พรรคประชาชาติ              43
พรรคครูไทยเพื่อประชาชน     28
พรรคพลังประชารัฐ           28
พรรคกล้าธรรม               24
พรรครวมไทยสร้างชาติ        24
พรรคพลังท้องถิ่นไท         13
พรรคชาติไทยพัฒนา           11
พรรคเสรีรวมไทย              5
พรรคเศรษฐกิจ                4
พรรคใหม่                    3
พรรคประชาธิปไตยใหม่         2
พรรคเป็นธรรม                2
พรรคท้องที่ไทย              2
พรรคพลังธรรมใหม่            1
พรรคไทยสร้างไทย             1
Name: party_cleaned, dtype: int64

In [21]:
df_cleaned.isnull().sum()

title              0
status             0
proposal_date      0
categories       666
creator_label      0
party_cleaned      0
dtype: int64

In [22]:
df_cleaned.shape[0]

666

In [23]:
df_cleaned.head(10)

,title,status,proposal_date,categories,creator_label,party_cleaned
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,เลขาธิการพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรครวมไทยสร้างชาติ,พรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,NaN,สมาชิกพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
10,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,NaN,สมาชิกพรรคประชาชน,พรรคประชาชน
13,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,NaN,หัวหน้าพรรคครูไทยเพื่อประชาชน,พรรคครูไทยเพื่อประชาชน
14,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,NaN,กรรมการบริหารพรรคกล้าธรรม,พรรคกล้าธรรม
15,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,NaN,สมาชิกพรรคครูไทยเพื่อประชาชน,พรรคครูไทยเพื่อประชาชน
18,ร่างพระราชบัญญัติการส่งเสริมวิทยาศาสตร์ การวิจ...,IN_PROGRESS,2025-09-02,NaN,สมาชิกพรรคเพื่อไทย,พรรคเพื่อไทย
21,ร่างพระราชบัญญัติโรงงาน (ฉบับที่ ..) พ.ศ. ....,IN_PROGRESS,2025-09-01,NaN,สมาชิกพรรคประชาชน,พรรคประชาชน
25,ร่างพระราชบัญญัติธรรมนูญศาลทหาร (ฉบับที่่ ..) ...,REJECTED,2025-07-30,NaN,สมาชิกพรรคประชาชาติ,พรรคประชาชาติ


In [24]:
# นับจำนวนแถวต่อ 1 title และเรียงจากมากไปน้อย
duplicate_counts = df_cleaned.groupby('title').size().sort_values(ascending=False)

# แสดงเฉพาะอันที่ซ้ำ (count > 1)
print(duplicate_counts[duplicate_counts > 1])

title
ร่างพระราชบัญญัติแก้ไขเพิ่มเติมประมวลกฎหมายอาญา (ฉบับที่ ..) พ.ศ. ....                                           18
ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดการประมง พ.ศ. 2558 พ.ศ. ....                                           17
ร่างพระราชบัญญัติการศึกษาแห่งชาติ พ.ศ. ....                                                                      15
ร่างพระราชบัญญัติลักษณะปกครองท้องที่ (ฉบับที่ ..) พ.ศ. ....                                                      12
ร่างพระราชบัญญัติอาสาสมัครสาธารณสุขประจำหมู่บ้าน พ.ศ. ....                                                       10
                                                                                                                 ..
ร่างพระราชบัญญัติการขนส่งทางราง พ.ศ. ....                                                                         2
พระราชบัญญัติควบคุมเครื่องดื่มแอลกอฮอล์ (ฉบับที่ ..) พ.ศ. ....                                                    2
ร่างพระราชบัญญัติตำรวจแห่งชาติ (ฉบับที่ ..) พ.ศ. ....             

In [25]:
!pip install -q -U google-generativeai


In [ ]:
# !pip install -q -U google-generativeai

import google.generativeai as genai
import pandas as pd
import time

genai.configure(api_key="")
model = genai.GenerativeModel('gemini-2.5-flash')

In [37]:
# 1. เตรียมรายชื่อร่างกฎหมายที่ไม่ซ้ำกันที่มีค่า null
unique_titles = df_cleaned[df_cleaned['categories'].isna()]['title'].unique().tolist()

def classify_titles(titles_list):
    results = {}
    # แบ่งทำเป็นชุดละ 20 ชื่อเพื่อไม่ให้ Prompt ยาวเกินไปและป้องกัน Error
    batch_size = 20
    
    for i in range(0, len(titles_list), batch_size):
        batch = titles_list[i:i + batch_size]
        
        prompt = f"""
        จงจัดหมวดหมู่ร่างกฎหมายไทยต่อไปนี้ โดยเลือกจากหมวดหมู่ที่กำหนด: 
        [การเมือง, เศรษฐกิจ, สวัสดิการ, สาธารณสุข, การศึกษา, สิ่งแวดล้อม, กระบวนการยุติธรรม, แรงงาน, วัฒนธรรม, อื่นๆ]
        
        ตอบในรูปแบบ JSON เท่านั้น: {{"ชื่อร่าง": "หมวดหมู่"}}
        
        รายชื่อร่างกฎหมาย:
        {batch}
        """
        
        try:
            response = model.generate_content(prompt)
            # ทำความสะอาด output และแปลงเป็น dict
            clean_response = response.text.replace('```json', '').replace('```', '').strip()
            batch_result = pd.io.json.loads(clean_response)
            results.update(batch_result)
            print(f"Processed {i + len(batch)} / {len(titles_list)}")
            time.sleep(1) # กัน API Rate limit
        except Exception as e:
            print(f"Error at batch {i}: {e}")
            
    return results

# 2. เริ่มการ Classify
predicted_map = classify_titles(unique_titles)

# 3. นำกลับไปเติมใน DataFrame
df_cleaned['categories'] = df_cleaned['categories'].fillna(df_cleaned['title'].map(predicted_map))

Processed 20 / 298
Processed 40 / 298
Processed 60 / 298
Processed 80 / 298
Processed 100 / 298
Processed 120 / 298
Processed 140 / 298
Processed 160 / 298
Processed 180 / 298
Processed 200 / 298
Processed 220 / 298
Processed 240 / 298
Processed 260 / 298
Processed 280 / 298
Processed 298 / 298


/var/folders/65/6rnggm5s6y5bnwcrgm55lbbw0000gn/T/ipykernel_41739/700063071.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['categories'] = df_cleaned['categories'].fillna(df_cleaned['title'].map(predicted_map))


In [38]:
df_cleaned['categories'].isna().sum()

13

In [39]:
df_cleaned.head()

,title,status,proposal_date,categories,creator_label,party_cleaned
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,เลขาธิการพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,สมาชิกพรรครวมไทยสร้างชาติ,พรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,สมาชิกพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
10,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,การเมือง,สมาชิกพรรคประชาชน,พรรคประชาชน
13,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,หัวหน้าพรรคครูไทยเพื่อประชาชน,พรรคครูไทยเพื่อประชาชน


In [40]:
df_cleaned['categories'].value_counts()

เศรษฐกิจ             168
การเมือง             115
กระบวนการยุติธรรม     84
การศึกษา              70
สิ่งแวดล้อม           64
สวัสดิการ             52
สาธารณสุข             43
วัฒนธรรม              32
แรงงาน                14
อื่นๆ                 11
Name: categories, dtype: int64

In [41]:
df_cleaned.shape[0]

666

In [50]:
remaining = df_cleaned[df_cleaned['categories'].isna()]['title'].unique().tolist()

for i, title in enumerate(remaining):
    print(f"'{title}'")

'ร่างพระราชบัญญัติยกเว้นความผิดให้แก่บุคคลที่ได้รับความเสียหายหรือได้รับผลกระทบ
จากการดำเนินการตามนโยบายของรัฐด้านป่าไม้และที่ดิน พ.ศ. ....'
'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 16/2560 
เรื่อง การบริหารงานบุคคลของข้าราชการครูและบุคลากรทางการศึกษา ลงวันที่ 21 มีนาคม พุทธศักราช 2560 พ.ศ. ....'
'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 เรื่อง 
การปฏิบัติหน้าที่ของคณะกรรมการคุรุสภา คณะกรรมการส่งเสริมสวัสดิการและสวัสดิภาพครู
และบุคลากรทางการศึกษา และคณะกรรมการบริหารองค์การค้าของสำนักงานคณะกรรมการ
ส่งเสริมสวัสดิการและสวัสดิภาพครูและบุคลากรทางการศึกษา ลงวันที่ 16 เมษายน พุทธศักราช
 2558 และคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 17/2560 เรื่อง แก้ไขเพิ่มเติม
คำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 ลงวันที่ 21 มีนาคม 2560 
พ.ศ. ....'
'ร่างพระราชบัญญัติคุณสมบัติมาตรฐานสำหรับกรรมการและพนักงานรัฐวิสาหกิจ (ฉบับที่..) 
พ.ศ. ....'
'ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดการให้ความช่วยเหลือทางการเงินแก่ผู้ประกอบ
วิสาหกิจที่ได้รับผ

In [51]:
manual_fix_set = {
    'ร่างพระราชบัญญัติยกเว้นความผิดให้แก่บุคคลที่ได้รับความเสียหายหรือได้รับผลกระทบจากการดำเนินการตามนโยบายของรัฐด้านป่าไม้และที่ดิน พ.ศ. ....': 'สิ่งแวดล้อม',
    'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 16/2560 เรื่อง การบริหารงานบุคคลของข้าราชการครูและบุคลากรทางการศึกษา ลงวันที่ 21 มีนาคม พุทธศักราช 2560 พ.ศ. ....': 'การศึกษา',
    'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 เรื่อง การปฏิบัติหน้าที่ของคณะกรรมการคุรุสภา คณะกรรมการส่งเสริมสวัสดิการและสวัสดิภาพครูและบุคลากรทางการศึกษา และคณะกรรมการบริหารองค์การค้าของสำนักงานคณะกรรมการส่งเสริมสวัสดิการและสวัสดิภาพครูและบุคลากรทางการศึกษา ลงวันที่ 16 เมษายน พุทธศักราช 2558 และคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 17/2560 เรื่อง แก้ไขเพิ่มเติมคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 ลงวันที่ 21 มีนาคม 2560 พ.ศ. ....': 'การศึกษา',
    'ร่างพระราชบัญญัติคุณสมบัติมาตรฐานสำหรับกรรมการและพนักงานรัฐวิสาหกิจ (ฉบับที่..) พ.ศ. ....': 'การเมือง',
    'ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดการให้ความช่วยเหลือทางการเงินแก่ผู้ประกอบวิสาหกิจที่ได้รับผลกระทบจากการระบาดของโรคติดเชื้อไวรัสโคโรนา 2019 พ.ศ. 2563  พ.ศ. ....': 'เศรษฐกิจ',
    'ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดให้อำนาจกระทรวงการคลังกู้เงินเพื่อแก้ไขปัญหา เยียวยา และฟื้นฟูเศรษฐกิจและสังคม ที่ได้รับผลกระทบจากการระบาดของโรคติดเชื้อไวรัสโคโรนา 2019 พ.ศ. 2563 (ฉบับที่ ..) พ.ศ. ....': 'เศรษฐกิจ'
}

# ใช้ map เพื่อเติมค่าลงในช่องที่ยังเป็น Null ของ df_cleaned
df_cleaned['categories'] = df_cleaned['categories'].fillna(df_cleaned['title'].map(manual_fix_set))

# เช็คผลลัพธ์สุดท้าย
print(f"ค่า Null ที่เหลือ: {df_cleaned['categories'].isna().sum()}")

ค่า Null ที่เหลือ: 11


/var/folders/65/6rnggm5s6y5bnwcrgm55lbbw0000gn/T/ipykernel_41739/2257540714.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['categories'] = df_cleaned['categories'].fillna(df_cleaned['title'].map(manual_fix_set))


In [52]:
remaining = df_cleaned[df_cleaned['categories'].isna()]['title'].unique().tolist()

for i, title in enumerate(remaining):
    print(f"'{title}'")

'ร่างพระราชบัญญัติยกเว้นความผิดให้แก่บุคคลที่ได้รับความเสียหายหรือได้รับผลกระทบ
จากการดำเนินการตามนโยบายของรัฐด้านป่าไม้และที่ดิน พ.ศ. ....'
'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 16/2560 
เรื่อง การบริหารงานบุคคลของข้าราชการครูและบุคลากรทางการศึกษา ลงวันที่ 21 มีนาคม พุทธศักราช 2560 พ.ศ. ....'
'ร่างพระราชบัญญัติยกเลิกคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 เรื่อง 
การปฏิบัติหน้าที่ของคณะกรรมการคุรุสภา คณะกรรมการส่งเสริมสวัสดิการและสวัสดิภาพครู
และบุคลากรทางการศึกษา และคณะกรรมการบริหารองค์การค้าของสำนักงานคณะกรรมการ
ส่งเสริมสวัสดิการและสวัสดิภาพครูและบุคลากรทางการศึกษา ลงวันที่ 16 เมษายน พุทธศักราช
 2558 และคำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 17/2560 เรื่อง แก้ไขเพิ่มเติม
คำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ ที่ 7/2558 ลงวันที่ 21 มีนาคม 2560 
พ.ศ. ....'
'ร่างพระราชบัญญัติคุณสมบัติมาตรฐานสำหรับกรรมการและพนักงานรัฐวิสาหกิจ (ฉบับที่..) 
พ.ศ. ....'
'ร่างพระราชบัญญัติแก้ไขเพิ่มเติมพระราชกำหนดการให้ความช่วยเหลือทางการเงินแก่ผู้ประกอบ
วิสาหกิจที่ได้รับผ

In [53]:
# สร้างฟังก์ชันเช็ค keyword ในชื่อร่างที่ยังเป็น null
def final_clean_nulls(row):
    if pd.isna(row['categories']):
        title = str(row['title'])
        if 'ป่าไม้' in title or 'ที่ดิน' in title:
            return 'สิ่งแวดล้อม'
        if 'คำสั่งหัวหน้าคณะรักษาความสงบแห่งชาติ' in title or 'ครู' in title:
            return 'การศึกษา'
        if 'รัฐวิสาหกิจ' in title:
            return 'การเมือง'
        if 'เงินกู้' in title or 'วิสาหกิจ' in title or 'เศรษฐกิจ' in title:
            return 'เศรษฐกิจ'
        if 'จราจร' in title:
            return 'กระบวนการยุติธรรม'
        if 'ทันตกรรม' in title or 'สาธารณสุข' in title:
            return 'สาธารณสุข'
    return row['categories']

# นำไปใช้กับ df_cleaned
df_cleaned['categories'] = df_cleaned.apply(final_clean_nulls, axis=1)

# ตรวจสอบผลลัพธ์สุดท้าย
remaining = df_cleaned['categories'].isna().sum()
print(f"จำนวนค่า Null ที่เหลือ: {remaining}")

if remaining > 0:
    print("รายชื่อที่ยังหลุดการ Clean:")
    print(df_cleaned[df_cleaned['categories'].isna()]['title'].unique())

จำนวนค่า Null ที่เหลือ: 0


/var/folders/65/6rnggm5s6y5bnwcrgm55lbbw0000gn/T/ipykernel_41739/2110083316.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['categories'] = df_cleaned.apply(final_clean_nulls, axis=1)


In [54]:
df_cleaned.isnull().sum()

title            0
status           0
proposal_date    0
categories       0
creator_label    0
party_cleaned    0
dtype: int64

In [55]:
df_cleaned['categories'].value_counts()

เศรษฐกิจ             172
การเมือง             116
กระบวนการยุติธรรม     85
การศึกษา              72
สิ่งแวดล้อม           68
สวัสดิการ             52
สาธารณสุข             44
วัฒนธรรม              32
แรงงาน                14
อื่นๆ                 11
Name: categories, dtype: int64

In [56]:
df_cleaned.head(20)

,title,status,proposal_date,categories,creator_label,party_cleaned
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,เลขาธิการพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,สมาชิกพรรครวมไทยสร้างชาติ,พรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,สมาชิกพรรคประชาธิปัตย์,พรรคประชาธิปัตย์
10,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,การเมือง,สมาชิกพรรคประชาชน,พรรคประชาชน
13,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,หัวหน้าพรรคครูไทยเพื่อประชาชน,พรรคครูไทยเพื่อประชาชน
14,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,กรรมการบริหารพรรคกล้าธรรม,พรรคกล้าธรรม
15,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,สมาชิกพรรคครูไทยเพื่อประชาชน,พรรคครูไทยเพื่อประชาชน
18,ร่างพระราชบัญญัติการส่งเสริมวิทยาศาสตร์ การวิจ...,IN_PROGRESS,2025-09-02,เศรษฐกิจ,สมาชิกพรรคเพื่อไทย,พรรคเพื่อไทย
21,ร่างพระราชบัญญัติโรงงาน (ฉบับที่ ..) พ.ศ. ....,IN_PROGRESS,2025-09-01,เศรษฐกิจ,สมาชิกพรรคประชาชน,พรรคประชาชน
25,ร่างพระราชบัญญัติธรรมนูญศาลทหาร (ฉบับที่่ ..) ...,REJECTED,2025-07-30,กระบวนการยุติธรรม,สมาชิกพรรคประชาชาติ,พรรคประชาชาติ


In [57]:
df_cleaned = df_cleaned.drop(columns="creator_label")

In [58]:
df_cleaned.head(20)

,title,status,proposal_date,categories,party_cleaned
0,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,พรรคประชาธิปัตย์
1,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,พรรครวมไทยสร้างชาติ
2,ร่างพระราชบัญญัติชั่วโมงปฏิบัติงานด้านสาธารณสุ...,IN_PROGRESS,2025-10-15,สาธารณสุข,พรรคประชาธิปัตย์
10,ร่างพระราชบัญญัติข้อมูลข่าวสารสาธารณะ พ.ศ. ....,IN_PROGRESS,2025-10-30,การเมือง,พรรคประชาชน
13,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,พรรคครูไทยเพื่อประชาชน
14,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,พรรคกล้าธรรม
15,ร่างพระราชบัญญัติสภาครูและบุคลากรทางการศึกษา (...,MERGED,2025-09-03,การศึกษา,พรรคครูไทยเพื่อประชาชน
18,ร่างพระราชบัญญัติการส่งเสริมวิทยาศาสตร์ การวิจ...,IN_PROGRESS,2025-09-02,เศรษฐกิจ,พรรคเพื่อไทย
21,ร่างพระราชบัญญัติโรงงาน (ฉบับที่ ..) พ.ศ. ....,IN_PROGRESS,2025-09-01,เศรษฐกิจ,พรรคประชาชน
25,ร่างพระราชบัญญัติธรรมนูญศาลทหาร (ฉบับที่่ ..) ...,REJECTED,2025-07-30,กระบวนการยุติธรรม,พรรคประชาชาติ


In [59]:
df_cleaned.tail(20)

,title,status,proposal_date,categories,party_cleaned
2339,ร่างพระราชบัญญัติคุ้มครองเด็กที่เกิดโดยอาศัยเท...,IN_PROGRESS,2025-12-11,สาธารณสุข,พรรคกล้าธรรม
2341,ร่างพระราชบัญญัติว่าด้วยค่าบริการพิเศษ พ.ศ. ....,REJECTED,2024-08-22,อื่นๆ,พรรคภูมิใจไทย
2345,ร่างพระราชบัญญัติระเบียบบริหารราชการเมืองพัทยา...,IN_PROGRESS,2025-09-10,การเมือง,พรรคประชาชน
2348,ร่างพระราชบัญญัติสภาตำบลและองค์การบริหารส่วนตำ...,IN_PROGRESS,2025-09-10,การเมือง,พรรคประชาชน
2351,ร่างพระราชบัญญัติเทศบาล (ฉบับที่ ..) พ.ศ. ....,IN_PROGRESS,2025-09-10,การเมือง,พรรคประชาชน
2353,ร่างพระราชบัญญัติการบริหารงานและการให้บริการภา...,IN_PROGRESS,2025-10-30,อื่นๆ,พรรคประชาชน
2356,ร่างพระราชบัญญัติสถานบริการ พ.ศ. ....,IN_PROGRESS,2025-09-24,การเมือง,พรรคเพื่อไทย
2358,ร่างพระราชบัญญัติระเบียบบริหารราชการกรุงเทพมหา...,IN_PROGRESS,2025-09-11,การเมือง,พรรคประชาชน
2362,ร่างพระราชบัญญัติการเลือกตั้งสมาชิกสภาท้องถิ่น...,IN_PROGRESS,2025-09-10,การเมือง,พรรคประชาชน
2364,ร่างพระราชบัญญัติส่งเสริมวัฒนธรรมสร้างสรรค์ พ....,REJECTED,2025-10-30,วัฒนธรรม,พรรคเพื่อไทย


In [65]:
df_cleaned.to_csv("bills_cleaned_and_labeled.csv")

AttributeError: 'str' object has no attribute 'to_csv'